# LoRA: adapters, gradients, and merging

**LoRA: Low-Rank Adaptation of Large Language Models** · Hu et al. · 2021

[Original paper](https://arxiv.org/abs/2106.09685)

**Learning goal:** Build and train a low-rank update, then verify merged-weight equivalence.

## Before starting

Training fundamentals, matrix products, and introductory LoRA.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

Fine-tuning need not update every full matrix. LoRA constrains updates to low rank while freezing base weights. This constrains the update family rather than merely compressing an already trained matrix.

Two smaller matrices reduce trainable parameters and optimizer states, but base-model and activation memory remain. We study one Linear; target-module selection and downstream quality require further experiments.

## Reproduction boundary

One Linear without dropout or quantization; not QLoRA, PEFT integration, or paper benchmarks.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`y = x W0^T + (alpha/r) x A^T B^T`, with `A:[r,in]`, `B:[out,r]`. Trainable count is `r*(in+out)`.

With B=0, initial output matches the base. On the first backward A normally has zero gradient and B a nonzero one. If both start at zero, both gradients vanish. Merge uses `Wmerged = W0 + (alpha/r) B A`.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Build the layer; verify initial equivalence and first-step gradient asymmetry.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, inputs=16, outputs=12, rank=3, alpha=6):
        super().__init__()
        self.base = nn.Linear(inputs, outputs, bias=False).requires_grad_(False)
        self.A = nn.Parameter(torch.randn(rank, inputs) * 0.02)
        self.B = nn.Parameter(torch.zeros(outputs, rank))
        self.scale = alpha / rank
    def forward(self, x):
        return self.base(x) + self.scale * (x @ self.A.T @ self.B.T)
layer = LoRALinear(); x = torch.randn(64, 16)
target = torch.randn(64, 12)
assert torch.equal(layer(x), layer.base(x))
F.mse_loss(layer(x), target).backward()
print('first A/B grad:', layer.A.grad.norm().item(), layer.B.grad.norm().item())
assert layer.A.grad.norm() == 0 and layer.B.grad.norm() > 0
assert layer.base.weight.grad is None

## Step 2

Fit a controlled low-rank target using only adapters. The target is deliberately low-rank; this verifies implementation, not superiority to full tuning.

In [ ]:
frozen = layer.base.weight.detach().clone()
target_update = torch.randn(12, 3) @ torch.randn(3, 16) * 0.1
target = F.linear(x, frozen + target_update)
optimizer = torch.optim.AdamW([layer.A, layer.B], lr=0.03, weight_decay=0)
losses = []
for step in range(160):
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(layer(x), target); loss.backward(); optimizer.step()
    losses.append(loss.item())
assert torch.equal(layer.base.weight, frozen)
assert losses[-1] < losses[0]
print('trainable:', layer.A.numel() + layer.B.numel(), 'full matrix:', frozen.numel())
plt.plot(losses, color='#d73535'); plt.xlabel('step'); plt.ylabel('adapter MSE'); plt.show()

## Step 3

Merging must preserve the function. Do not add the adapter again on top of merged weights.

In [ ]:
with torch.no_grad():
    merged_weight = layer.base.weight + layer.scale * (layer.B @ layer.A)
    adapter_output = layer(x)
    merged_output = F.linear(x, merged_weight)
error = (adapter_output - merged_output).abs().max().item()
print('merge error:', error)
assert torch.allclose(adapter_output, merged_output, atol=1e-5)
bad_A = nn.Parameter(torch.zeros(3, 16)); bad_B = nn.Parameter(torch.zeros(12, 3))
(x @ bad_A.T @ bad_B.T - target).square().mean().backward()
assert bad_A.grad.norm() == 0 and bad_B.grad.norm() == 0

## Reading the result

Loss should fall because the target is a rank-3 update. The frozen-base assertion detects accidental full tuning. Merge uses a tolerance because operation order changes float32 rounding.

## Independent experiments

1. Use rank=1 for the rank-3 target.
2. Increase alpha without changing learning rate.
3. Merge twice and compare outputs.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

A rank-1 adapter generally cannot represent a rank-3 update. Alpha scales the effective update. Merging twice doubles the delta and changes the function: merging must be state-aware.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.